In [1]:
import torch
import numpy as np
import normflows as nf

import sys
import os
c_directory = os.getcwd()
sys.path.append(os.path.dirname(c_directory))
# sys.path.append(os.path.join(os.path.dirname(c_directory), 'FCYeast'))

from tqdm import tqdm
import architecture
import SCD_simulator
#import eZplot
from SCD_plot import SCD_plot

from matplotlib import pyplot as plt
from tqdm import tqdm


enable_cuda = True
CUDA_LAUNCH_BLOCKING=1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SCD_simulator.adjust_device(device)


In [2]:
# Define flows
model_file = 'network.pt'
figs_direc = 'network_perform'

model = architecture.make_model(context_size=2,tail_bound=15)

In [3]:
# Define target
SCD_simulator.adjust_device(device)
target = SCD_simulator.target()

In [4]:
try:
    model.load_state_dict(torch.load(model_file))
    print('loading pretrained network')
except:
    print('starting from scratch')

loading pretrained network


/tmp/ipykernel_12288/3586049576.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_file))


In [5]:
max_iter   = 2000
show_iter  = 100
n_batch    = 16

x = target.sample(N=1024*1024)
batch_size = x.size(0)//n_batch

x,context = x[:,0].reshape(-1,1)*1.0,x[:,1:]

batches = torch.arange(x.size(0)).reshape(n_batch,-1)

In [6]:
# Train model
loss_hist = np.array([])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-6)

In [7]:
for it in tqdm(range(max_iter)):
    loss_epoch = np.array([])

    for it2 in (range(n_batch)):  
        optimizer.zero_grad() 
        # Compute loss
        batch = batches[it2]
        loss = -model.log_prob(x[batch], context[batch]).mean()
        
        # Do backprop and optimizer step
        if ~(torch.isnan(loss) | torch.isinf(loss)):
            loss.backward()
            optimizer.step()
        
        loss_epoch = np.append(loss_epoch, loss.to('cpu').item())

    # Log loss
    loss_hist = np.append(loss_hist, np.mean(loss_epoch))

    if (it+1)%show_iter==0:
        index = (loss_hist.size+np.arange(-int(2.5*show_iter),0))

        
        with torch.no_grad():
            if len(loss_hist) >= show_iter:
                if loss_hist[-1] < loss_hist[-show_iter]:
                    torch.save(model.state_dict(), model_file)
                else:
                    #if the model got worse, go back to last and resample
                    model.load_state_dict(torch.load(model_file))
            else:
                torch.save(model.state_dict(), model_file)
        SCD_plot(model,target,loss_hist,index[index>=0])

    else:
        with torch.no_grad():
            samples_new = target.sample(N=16*1024)

            x[:samples_new.size(0)] = samples_new[:,0].reshape((-1,1))
            context[:samples_new.size(0)] = samples_new[:,1:]

            shuffle_index = torch.randperm(x.size(0))
            x = x[shuffle_index]
            context = context[shuffle_index]

 25%|██▍       | 499/2000 [39:35<1:54:50,  4.59s/it]/tmp/ipykernel_12288/1959886346.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(mod

In [8]:
torch.save(model.state_dict(), model_file)
np.savetxt(model_file.split('.')[0]+'_loss_hist.csv',loss_hist)
